# Fine-Tune LLaMA 3.2 3B for House Price Prediction

**Run this notebook in Google Colab with a T4 GPU**

This notebook fine-tunes `meta-llama/Llama-3.2-3B-Instruct` using QLoRA (4-bit quantization + LoRA adapters) for house price prediction.

**Prerequisites:**
1. HuggingFace account with access to LLaMA 3.2 models
2. W&B account (project: `llama-3.2-housing-pricer`)
3. Training data file: `train_finetune.jsonl` (generated from main notebook)

## Step 1: Install Dependencies

In [1]:
!pip install -q transformers peft bitsandbytes accelerate trl datasets huggingface_hub wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 39.9 MB/s eta 0:00:00


## Step 2: Login to HuggingFace and W&B

In [2]:
from huggingface_hub import login
login()

In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 ae64c4bc9e1b59db383d5892e2dbde1ca2dec0d8


wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: davenmathews (davenjeru) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Step 3: Upload Training Data

Upload `train_finetune.jsonl` from your local machine, or run the cell below to create sample data.

In [4]:
# Option 1: Upload from local machine
from google.colab import files
uploaded = files.upload()  # Upload train_finetune.jsonl

Saving train_finetune.jsonl to train_finetune (1).jsonl


In [5]:
# # Option 2: Create sample training data (for testing)
# import json

# SYSTEM_PROMPT = "You are a real estate appraiser. Based on property details, estimate the market price."

# sample_data = [
#     {"description": "A 3 bedroom, 2 bathroom house with 1,850 square feet on a 0.25 acre lot located in Austin, Texas, 78701.", "price": 450000},
#     {"description": "A 4 bedroom, 3 bathroom house with 2,500 square feet on a 0.5 acre lot located in Denver, Colorado, 80202.", "price": 650000},
#     {"description": "A 2 bedroom, 1 bathroom house with 1,200 square feet located in Phoenix, Arizona, 85001.", "price": 280000},
#     {"description": "A 5 bedroom, 4 bathroom house with 3,500 square feet on a 1.0 acre lot located in Miami, Florida, 33101.", "price": 950000},
#     {"description": "A 3 bedroom, 2 bathroom house with 1,600 square feet located in Seattle, Washington, 98101.", "price": 520000},
# ]

# def format_for_finetuning(desc, price):
#     return {
#         "text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
# {SYSTEM_PROMPT}<|eot_id|>
# <|start_header_id|>user<|end_header_id|>
# What is the estimated price of this property?

# {desc}<|eot_id|>
# <|start_header_id|>assistant<|end_header_id|>
# ${price:,}<|eot_id|>"""
#     }

# with open('train_finetune.jsonl', 'w') as f:
#     for item in sample_data:
#         f.write(json.dumps(format_for_finetuning(item['description'], item['price'])) + '\n')

# print("Created sample training data with 5 examples")

Created sample training data with 5 examples


## Step 4: Load Model with 4-bit Quantization

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "meta-llama/Llama-3.2-3B-Instruct"

print(f"Loading {model_name} with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded. Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

Loading meta-llama/Llama-3.2-3B-Instruct with 4-bit quantization...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Model loaded. Memory footprint: 2.20 GB


## Step 5: Configure LoRA Adapters

In [16]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

Trainable parameters: 9,175,040 (0.51%)


## Step 6: Load Training Dataset

In [17]:
dataset = load_dataset('json', data_files='train_finetune.jsonl', split='train')
print(f"Loaded {len(dataset)} training examples")
print("\nExample:")
print(dataset[0]['text'][:300] + "...")

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 510 training examples

Example:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a real estate appraiser. Based on property details, estimate the market price.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
What is the estimated price of this property?

A 3 bedroom, 2.0 bathroom house with 1,401 square feet ...


## Step 7: Train with W&B Logging

In [ ]:
wandb.init(project="llama-3.2-housing-pricer", name="llama-3.2-3b-qlora")

training_args = SFTConfig(
    output_dir="./llama-house-pricer",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    report_to="wandb",
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()

TRL version: 0.29.0


Adding EOS to train dataset:   0%|          | 0/510 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/510 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/510 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Starting training...


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

## Step 8: Save Model

In [ ]:
model.save_pretrained("./llama-house-pricer-final")
tokenizer.save_pretrained("./llama-house-pricer-final")
print("Model saved to ./llama-house-pricer-final")

## Step 9: Test the Fine-Tuned Model

In [ ]:
SYSTEM_PROMPT = """
You are a real estate appraiser. Based on property details, estimate the market price.
You only respond with the price, no other text.
"""

test_description = "A 3 bedroom, 2.5 bathroom house with 2,100 square feet on a 0.3 acre lot located in Portland, Oregon, 97201."

prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{SYSTEM_PROMPT}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
What is the estimated price of this property?

{test_description}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Property: {test_description}")
print(f"\nPredicted price: {response.split('assistant')[-1].strip()}")

In [ ]:
wandb.finish()
print("\nTraining complete! Check your results at: https://wandb.ai/")

## Step 10: Download Model (Optional)

Download the fine-tuned model to your local machine.

In [ ]:
# Zip and download the model
!zip -r llama-house-pricer-final.zip ./llama-house-pricer-final

from google.colab import files
files.download('llama-house-pricer-final.zip')